# F-S-0-SUPP: Reconstruction Error vs. Number of Microphones

Single-frequency, noiseless sparse support-recovery setup. This plot shows coefficient reconstruction error as the number of microphones varies.

### 0. Imports

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start=Path.cwd()):
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "src" / "cs_priors").exists():
            return path
    raise RuntimeError("Could not find repository root")


REPO_ROOT = find_repo_root()
sys.path[:0] = [
    str(REPO_ROOT / "src"),
    str(REPO_ROOT / "notebooks" / "benchmarks_v2" / "functions"),
]

from figures import save_pdf
from single_frequency_benchmarks import (
    METHOD_LABELS,
    METHOD_ORDER,
    make_simulation,
    plot_metric,
    relabel_legend,
    run_microphone_count_benchmark,
    solve_methods,
)
from sanity import run_sanity_check
from single_frequency_config import (
    FULL_RING_SPAN,
    LASSO_ALPHA,
    LASSO_MAX_ITER,
    PHASE_SEEDS,
    base_sim_kwargs,
    sector_span_deg,
)


### 1.Parameters

In [ ]:
TAG = "F-S-0-SUPP"
FIGURE_DIR = REPO_ROOT / "results" / "benchmarks" / "v2" / TAG
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

NUM_SOURCES = 10
SECTOR_SPAN_DEG = sector_span_deg(NUM_SOURCES)
NUM_MICS_VALUES = np.arange(2, NUM_SOURCES + 1)
CHECK_NUM_MICS = 3
METHODS_TO_PLOT = METHOD_ORDER

SIM_KWARGS = base_sim_kwargs(
    num_sources=NUM_SOURCES,
    mic_angle_span=FULL_RING_SPAN,
    source_angle_span_deg=SECTOR_SPAN_DEG,
)

# To showcase temp_arc_array
# SIM_KWARGS["mic_angle_span"] = np.deg2rad(SECTOR_SPAN_DEG)

### 2. Sanity check

In [ ]:
sim = make_simulation(
    SECTOR_SPAN_DEG,
    PHASE_SEEDS[0],
    num_mics=CHECK_NUM_MICS,
    **SIM_KWARGS,
)
solutions = solve_methods(sim, lasso_alpha=LASSO_ALPHA, lasso_max_iter=LASSO_MAX_ITER)

run_sanity_check(
    sim,
    solutions=solutions,
    save_path=FIGURE_DIR / "simulation_geometry_num_mics_check.pdf",
    
    # view_limits=((-0.03, 0.35), (-0.03, 0.22)),
)

### 3. Generating results

In [ ]:
results_df = run_microphone_count_benchmark(
    NUM_MICS_VALUES,
    PHASE_SEEDS,
    separation_deg=SECTOR_SPAN_DEG,
    sim_kwargs=SIM_KWARGS,
    lasso_alpha=LASSO_ALPHA,
    lasso_max_iter=LASSO_MAX_ITER,
)

results_df["method"] = pd.Categorical(
    results_df["method"], categories=METHODS_TO_PLOT, ordered=True
)
results_df = results_df.sort_values(["num_mics", "phase_seed", "method"]).reset_index(drop=True)
results_df.head()

### 3.5 Example source amplitudes

In [ ]:
def plot_example(EXAMPLE_NUM_MICS, EXAMPLE_PHASE_SEED=0, both=False):

    example_sim = make_simulation(
        SECTOR_SPAN_DEG,
        EXAMPLE_PHASE_SEED,
        num_mics=EXAMPLE_NUM_MICS,
        **SIM_KWARGS,
    )
    example_solutions = solve_methods(
        example_sim,
        lasso_alpha=LASSO_ALPHA,
        lasso_max_iter=LASSO_MAX_ITER,
    )

    amplitudes = {
        r"$\boldsymbol{x}_{true}$": np.abs(example_sim.X[:, 0]),
        METHOD_LABELS["X_pinv"]: np.abs(example_solutions["X_pinv"][:, 0]),
        METHOD_LABELS["r-LASSO"]: np.abs(example_solutions["r-LASSO"][:, 0]),
        METHOD_LABELS["Sparse Prior"]: np.abs(example_solutions["Sparse Prior"][:, 0]),
    }

    source_index = np.arange(example_sim.X.shape[0])
    bar_width = 0.20
    offsets = (np.arange(len(amplitudes)) - (len(amplitudes) - 1) / 2) * bar_width
    if both:
        fig, ax = plt.subplots(figsize=(7.2, 4.0))
        for offset, (label, amplitude) in zip(offsets, amplitudes.items()):
            ax.bar(source_index + offset, amplitude, width=bar_width, label=label)

        ax.set_xticks(source_index)
        ax.set_xlabel("Source index")
        ax.set_ylabel("Single-frequency coefficient amplitude")
        ax.set_title(f"Example source amplitudes, M = {EXAMPLE_NUM_MICS}, seed = {EXAMPLE_PHASE_SEED}")
        ax.grid(True, axis="y", linewidth=0.5, alpha=0.35)
        ax.legend(frameon=False, ncols=2)
        fig.tight_layout()

        save_pdf(fig, FIGURE_DIR / "source_amplitudes_example.pdf")
        plt.show()

    bar_colors = np.array(["tab:blue"] * len(source_index), dtype=object)
    bar_colors[list(example_sim.active_indices)] = "gold"

    fig, axes = plt.subplots(2, 2, figsize=(7.2, 5.2), sharex=True, sharey=True)
    axes = axes.ravel()

    for ax, (label, amplitude) in zip(axes, amplitudes.items()):
        ax.bar(source_index, amplitude, color=bar_colors, edgecolor="black", linewidth=0.3)
        ax.set_title(label)
        ax.grid(True, axis="y", linewidth=0.5, alpha=0.35)

    for ax in axes[-2:]:
        ax.set_xlabel("Source index")
    for ax in axes[::2]:
        ax.set_ylabel("Amplitude")

    fig.suptitle(f"Example source amplitudes, M = {EXAMPLE_NUM_MICS}, seed = {EXAMPLE_PHASE_SEED}")
    fig.tight_layout()

    save_pdf(fig, FIGURE_DIR / "source_amplitudes_example_subplots.pdf")
    plt.show()


    pd.DataFrame(
        {
            "method": list(example_solutions),
            "relative_complex_error": [
                np.linalg.norm(X_hat - example_sim.X) / np.linalg.norm(example_sim.X)
                for X_hat in example_solutions.values()
            ],
        }
    )
plot_example(6)

### 4. Plotting results

In [ ]:
fig, ax, summary_df = plot_metric(
    results_df,
    x_col="num_mics",
    method_order=METHODS_TO_PLOT,
    xlabel="Number of microphones",
    title="Reconstruction error vs. number of microphones",
    xscale="linear",
    yscale="linear",
)
relabel_legend(ax, METHOD_LABELS)
save_pdf(fig, FIGURE_DIR / "relative_complex_error_vs_num_mics.pdf")
plt.show()

summary_df.head()

In [ ]:
fig, ax, f1_summary_df = plot_metric(
    results_df,
    x_col="num_mics",
    y_col="f1_threshold_10_percent",
    method_order=METHODS_TO_PLOT,
    xlabel="Number of microphones",
    ylabel=r"$F_1$-score",
    title=r"$F_1$-score vs. number of microphones",
    xscale="linear",
    yscale="linear",
)
ax.set_ylim(-0.02, 1.02)
relabel_legend(ax, METHOD_LABELS)
save_pdf(fig, FIGURE_DIR / "f1_threshold_10_percent_vs_num_mics.pdf")
plt.show()

f1_summary_df.head()